# Тестовый скрипт

In [8]:
import sys
print("Интерпретатор:", sys.executable)

try:
    import speechbrain
    print("SpeechBrain успешно импортирован! Версия:", speechbrain.__version__)
    from speechbrain.inference.classifiers import EncoderClassifier
    print("EncoderClassifier из inference доступен!")
except Exception as e:
    print("Ошибка:", type(e).__name__, e)

Интерпретатор: C:\Users\Svetozar\PycharmProjects\ASTRA_voiceprint\.venv\Scripts\python.exe
SpeechBrain успешно импортирован! Версия: 1.1.1
EncoderClassifier из inference доступен!


# Старт

In [9]:
import os
import torch
import torchaudio
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from speechbrain.inference.classifiers import EncoderClassifier
from speechbrain.utils.fetching import LocalStrategy

if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")

print("Текущая папка проекта:", os.getcwd())

dev_dir = os.path.abspath("data/raw/developer")
dist_dir = os.path.abspath("data/raw/distractors")

dev_files = [os.path.join(dev_dir, f) for f in os.listdir(dev_dir) if f.lower().endswith(".wav")] if os.path.exists(dev_dir) else []
dist_files = [os.path.join(dist_dir, f) for f in os.listdir(dist_dir) if f.lower().endswith(".wav")] if os.path.exists(dist_dir) else []

print(f"Найдено записей создателя: {len(dev_files)}")
print(f"Найдено записей других людей: {len(dist_files)}")

if len(dev_files) == 0 or len(dist_files) == 0:
    raise FileNotFoundError(
        "Положи хотя бы по 2-3 .wav файла в 'data/raw/developer' и 'data/raw/distractors' перед построением графика!"
    )

device = "cuda" if torch.cuda.is_available() else "cpu"

classifier = EncoderClassifier.from_hparams(
    source="models/teacher/ecapa_voxceleb",
    savedir="models/teacher/ecapa_voxceleb",
    run_opts={"device": device},
    local_strategy=LocalStrategy.COPY
)

def get_embedding(wav_path):
    signal, fs = torchaudio.load(wav_path)
    if fs != 16000:
        signal = torchaudio.transforms.Resample(fs, 16000)(signal)
    with torch.no_grad():
        emb = classifier.encode_batch(signal.to(device))
        emb = torch.nn.functional.normalize(emb, dim=2).squeeze().cpu().numpy()
    return emb

all_files = dev_files + dist_files
labels = [f"Dev_{i+1}" for i in range(len(dev_files))] + [f"Other_{i+1}" for i in range(len(dist_files))]

embeddings = [get_embedding(f) for f in all_files]

matrix = np.zeros((len(all_files), len(all_files)))
for i in range(len(all_files)):
    for j in range(len(all_files)):
        matrix[i, j] = np.dot(embeddings[i], embeddings[j])

plt.figure(figsize=(8, 6))
sns.heatmap(matrix, xticklabels=labels, yticklabels=labels, annot=True, cmap="mako", fmt=".2f")
plt.title("Матрица косинусного сходства (Teacher: ECAPA-TDNN)")
plt.show()

Текущая папка проекта: C:\Users\Svetozar\PycharmProjects\ASTRA_voiceprint
Найдено записей создателя: 0
Найдено записей других людей: 0


FileNotFoundError: Положи хотя бы по 2-3 .wav файла в 'data/raw/developer' и 'data/raw/distractors' перед построением графика!

# Результаты